# Teste manual: protocolos multiagente no GSM8K

Roda os modelos reais (`Qwen2.5-3B-Instruct` como minion e `Qwen2.5-14B-Instruct`
como mestre, via HuggingFace Transformers na GPU) sobre uma amostra pequena de
perguntas reais do GSM8K, para inspecionar pergunta a pergunta o que cada
protocolo respondeu antes de rodar o experimento completo.

In [1]:
%load_ext autoreload
%autoreload 2
# Com autoreload ativo, edições em config.py e src/*.py são recarregadas
# automaticamente — não é preciso reiniciar o kernel (e portanto não é
# preciso recarregar os modelos na GPU) a cada ajuste no código.

import os
import sys

# Garante backend real (HuggingFace Transformers na GPU) — não usar mock aqui.
# Precisa ser definido antes do primeiro `import torch` (feito sob demanda
# em src/llm.py).
os.environ["MULTIAGENT_BACKEND"] = "hf"

ROOT = os.path.abspath(".")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [2]:
import torch
from tqdm.notebook import tqdm

import config as cfg
from src.dataset import load_samples
from src.llm import LLMHub
from src.metrics import aggregate, is_correct
from src.protocols import available, get_protocol

print("GPUs visíveis:", torch.cuda.device_count())
print(f"Backend: {cfg.BACKEND} | Minion: {cfg.MINION_MODEL.label} ({cfg.MINION_MODEL.name})")
print(f"Mestre: {cfg.MASTER_MODEL.label} ({cfg.MASTER_MODEL.name})")
print("Protocolos disponíveis:", available())

GPUs visíveis: 1
Backend: hf | Minion: Qwen2.5-3B (Qwen/Qwen2.5-3B-Instruct)
Mestre: Qwen2.5-14B (Qwen/Qwen2.5-14B-Instruct)
Protocolos disponíveis: ['debate', 'minions', 'single_agent', 'single_minion']


## Amostra do GSM8K

In [3]:
N_SAMPLES = 5  # amostra pequena só para inspecionar as respostas
SEED = 42

samples = load_samples(N_SAMPLES, SEED, cfg.EXPERIMENT.dataset, cfg.EXPERIMENT.split)
print(f"{len(samples)} perguntas reais do GSM8K carregadas.")

5 perguntas reais do GSM8K carregadas.


In [4]:
for i, s in enumerate(samples, 1):
    print(f"[{i}] {s.question}\n    gold = {s.gold}\n")

[1] Jared is trying to increase his typing speed. He starts with 47 words per minute (WPM). After some lessons the next time he tests his typing speed it has increased to 52 WPM. If he continues to increase his typing speed once more by 5 words, what will be the average of the three measurements?
    gold = 52

[2] Jordan has 2 children who wear diapers.  Each child requires 5 diaper changes per day.  Jordan's wife changes half of the diapers.  How many diapers does Jordan change per day?
    gold = 5

[3] A wooden bridge can carry no more than 5000 pounds. A delivery truck filled with identical boxes, each weighing 15 pounds, will pass over the bridge. The combined weight of the driver and the empty truck is 3755 pounds. What is the maximum number of boxes which can be loaded onto the truck while not exceeding the bridge's weight limit?
    gold = 83

[4] Tim has a box with 7 blue shoe boxes and 9 red shoe boxes. If he uses 3 blue shoeboxes and 1/3 red of his shoeboxes to go fishing, 

## Modelos

Carregados sob demanda (lazy) na primeira chamada e mantidos em cache pelo
`LLMHub`. Execute a célula abaixo **uma única vez por sessão**: com o
`autoreload` ativado na célula de setup, qualquer edição em `config.py` ou
`src/` é recarregada automaticamente, então não há motivo para recriar o hub
(e re-carregar Qwen2.5-3B/14B na GPU) entre uma simulação e outra.

In [5]:
# Reaproveita o hub se a célula for executada de novo por engano — assim os
# modelos (já carregados na GPU) nunca são recarregados sem necessidade.
if "hub" not in globals():
    hub = LLMHub()
    print("Hub criado — modelos serão carregados sob demanda na 1ª chamada.")
else:
    print("Hub já existe nesta sessão — reaproveitando (nada foi recarregado).")

Hub criado — modelos serão carregados sob demanda na 1ª chamada.


## Simulações por protocolo

Cada protocolo roda em sua própria célula, usando o mesmo `hub` (modelos já
carregados) e a mesma `samples`. Rode só a célula do protocolo que quiser
inspecionar — não é preciso re-rodar as outras nem recriar o hub.

In [6]:
# nome do protocolo -> list[QueryResult]. Mantido entre execuções: rodar um
# protocolo de novo só sobrescreve a entrada dele, sem afetar os demais.
if "all_results" not in globals():
    all_results = {}

In [7]:
def run_protocol(name: str):
    proto = get_protocol(name, hub)
    results = []
    print(f"\n▶ Protocolo: {name}")
    for s in tqdm(samples, desc=name, unit="q"):
        r = proto.run(s)
        results.append(r)
        status = "✅" if r.correct else "❌"
        print(f"  {status} pred={r.prediction!r:>10}  gold={r.gold:>10}  "
              f"master={'sim' if r.used_master else 'não'}  chamadas={r.n_model_calls}")
    all_results[name] = results
    return results

In [8]:
_ = run_protocol("single_minion")


▶ Protocolo: single_minion


single_minion:   0%|          | 0/5 [00:00<?, ?q/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  ✅ pred='Primeiro, vamos calcular o aumento na velocidade de 47 WPM para 52 WPM:\n52 - 47 = 5 WPM\n\nAgora, se ele aumentar mais 5 WPM, sua nova velocidade será:\n52 + 5 = 57 WPM\n\nCom os três valores: 47, 52 e 57 WPM, vamos calcular a média:\n(47 + 52 + 57) / 3 = 156 / 3 = 52 WPM\n\n#### 52'  gold=        52  master=não  chamadas=1
  ✅ pred='Para resolver este problema, vamos seguir os passos abaixo:\n\n1. Primeiro, calculamos o total de diários necessários por dia para as duas crianças:\n   - Cada criança precisa de 5 diários por dia.\n   - Então, para duas crianças: 5 + 5 = 10 diários por dia.\n\n2. Agora, sabemos que a esposa de Jordan alterna as trocas de diário. Isso significa que ela troca metade dos diários:\n   - Metade de 10 diários é 10 / 2 = 5 diários.\n\n3. Portanto, Jordan deve trocar os outros 5 diários.\n\n#### 5\n#### 5'  gold=         5  master=não  chamadas=1
  ✅ pred='Primeiro, calculamos o peso máximo que ainda pode ser adicionado ao veículo sem exceder o limite 

In [9]:
_ = run_protocol("single_agent")


▶ Protocolo: single_agent


single_agent:   0%|          | 0/5 [00:00<?, ?q/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

  ✅ pred="To find the average of Jared's three typing speed measurements, we need to follow these steps:\n\n1. Identify the initial typing speed.\n2. Identify the second typing speed after the lessons.\n3. Calculate the third typing speed after another increase.\n4. Compute the average of the three speeds.\n\nLet's start with the given values:\n- Initial typing speed: 47 WPM\n- Typing speed after lessons: 52 WPM\n- Increase in speed for the third measurement: 5 WPM\n\nFirst, calculate the third typing speed:\n\\[ \\text{Third typing speed} = 52 + 5 = 57 \\text{ WPM} \\]\n\nNext, we need to find the average of the three typing speeds:\n\\[ \\text{Average} = \\frac{\\text{Initial speed} + \\text{Second speed} + \\text{Third speed}}{3} \\]\n\\[ \\text{Average} = \\frac{47 + 52 + 57}{3} \\]\n\nNow, perform the addition inside the numerator:\n\\[ 47 + 52 + 57 = 156 \\]\n\nThen, divide by 3 to find the average:\n\\[ \\text{Average} = \\frac{156}{3} = 52 \\]\n\nThus, the average of the three 

In [10]:
_ = run_protocol("minions")


▶ Protocolo: minions


minions:   0%|          | 0/5 [00:00<?, ?q/s]

  ✅ pred='Primeiro, vamos calcular a média dos três valores: 47, 52 e 57 (que é 52 + 5).\n\nA média é dada pela soma dos valores dividida pelo número de valores.\n\nSoma = 47 + 52 + 57 = 156\n\nNúmero de valores = 3\n\nMédia = 156 / 3 = 52\n\n#### 52'  gold=        52  master=não  chamadas=1
  ✅ pred='Primeiro, vamos calcular quantas mudanças de fralda são necessárias por dia para as duas crianças:\n- Cada criança requer 5 mudanças de fralda por dia.\n- Então, para duas crianças, serão 5 + 5 = 10 mudanças de fralda por dia.\n\nSegundo, sabemos que a esposa de Jordan cuida metade dessas mudanças:\n- Portanto, ela cuida de 10 / 2 = 5 mudanças de fralda por dia.\n\nFinalmente, podemos determinar quantas mudanças de fralda o próprio Jordan deve fazer:\n- Como foram feitas 5 mudanças de fralda e essas foram feitas pela esposa, o número de mudanças de fralda que o Jordan deve fazer é 10 - 5 = 5.\n\n#### 5'  gold=         5  master=não  chamadas=1
  ✅ pred='Primeiro, vamos calcular o peso máx

In [ ]:
_ = run_protocol("mixture_of_agents")

In [ ]:
_ = run_protocol("debate")

## Resultados agregados

In [12]:
aggs = [aggregate(name, results) for name, results in all_results.items()]

print(f"{'Protocolo':<16}{'Acurácia':>10}{'Latência(s)':>14}"
      f"{'Tokens':>12}{'Tok.Mestre':>12}{'Uso Mestre':>12}{'Chamadas':>10}")
print("-" * 86)
for a in aggs:
    print(f"{a.protocol:<16}{a.accuracy:>9.1%}{a.avg_latency_s:>14.2f}"
          f"{a.avg_total_tokens:>12.0f}{a.avg_master_tokens:>12.0f}"
          f"{a.master_usage_rate:>11.1%}{a.avg_model_calls:>10.2f}")

Protocolo         Acurácia   Latência(s)      Tokens  Tok.Mestre  Uso Mestre  Chamadas
--------------------------------------------------------------------------------------
single_minion       80.0%          6.05         248           0       0.0%      1.00
single_agent       100.0%         13.31         362         257     100.0%      1.00
minions            100.0%          6.52         335           0       0.0%      1.00
debate              80.0%         52.13        2993         125     100.0%      5.00
